In [0]:
%run ../init_framework_silver

In [0]:
# 1. SOURCE INGESTION
# Read from Bronze CDF.
df_bronze_repayments = read_delta_stream(
    spark=spark, 
    table_name=REPAYMENTS_BRONZE
)    

In [0]:
# 2. SCHEMA MAPPING
repayments_rename_map = {
    "total_rec_prncp": "total_principal_received",
    "total_rec_int": "total_interest_received",
    "total_rec_late_fee": "total_late_fee_received",
    "total_pymnt": "total_payment_received",
    "last_pymnt_amnt": "last_payment_amount",
    "last_pymnt_d": "last_payment_date",
    "next_pymnt_d": "next_payment_date",
}

# Standardizing col names to match the Silver layer DDL
df_renamed_repayments = standardize_column_names(
    df_bronze_repayments, repayments_rename_map, strict=True
)

In [0]:
# 4. METADATA ENRICHMENT
# Injects standard audit columns (_ingested_at, etc.)
df_with_metadata = apply_silver_metadata(df_renamed_repayments)


In [0]:
# 5. DATA QUALITY: CRITICAL NULLS
# Drop records missing core ledger values; we cannot calculate risk without these.
col_ls = ["total_principal_received", "total_interest_received", "total_late_fee_received", "total_payment_received", "last_payment_amount"]
df_valid_repayments = drop_critical_nulls(df_with_metadata, col_ls)


In [0]:
# 6. REPAIR LOGIC: LEDGER CONSISTENCY
from pyspark.sql.functions import col, when

# Fix for total_payment_received where it might be 0.0 despite principal being received.
# Re-calculates the sum of components to ensure the ledger balances.
df_payment_fixed = df_valid_repayments.withColumn(
    "total_payment_received",
    when(
        (col("total_principal_received") != 0.0)
        & (col("total_payment_received") == 0.0),
        col("total_principal_received")
        + col("total_interest_received")
        + col("total_late_fee_received"),
    ).otherwise(col("total_payment_received")),
)

# Filter out records where no payment was actually recorded
df_valid_payment = df_payment_fixed.filter("total_payment_received != 0.0")


In [0]:
# 7. TRANSFORMATION: DATE CLEANING
from pyspark.sql.functions import col, when, lit

# Upstream "0.0" strings for dates are converted back to true NULLs.
# This ensures downstream date-time parsing doesn't fail.
df_repayments_final = df_valid_payment.withColumn(
    "last_payment_date",
    when(col("last_payment_date") == "0.0", lit(None).cast("string"))
    .otherwise(col("last_payment_date"))
).withColumn(
    "next_payment_date",
    when(col("next_payment_date") == "0.0", lit(None).cast("string"))
    .otherwise(col("next_payment_date"))
)


In [0]:
# 8. UPSERT LOGIC & STREAM EXECUTION
spark.conf.set("spark.sql.shuffle.partitions", "16")

def upsert_repayments_to_silver(microBatchDF, batchId):
    """
    Stateless MERGE into Silver Repayments.
    Uses _bronze_version check to handle high-frequency ledger updates correctly.
    """
    spark_session = microBatchDF.sparkSession
    
    from pyspark.sql import Window
    from pyspark.sql.functions import col, row_number, desc
        
    # Filter for valid CDC events (insert or post-image)
    df_updated = microBatchDF.filter(col("_change_type").isin("insert", "update_postimage"))

    # 1. Define the Window
    # Partition by the Primary Key and order by timestamp descending.
    # This ensures the freshest CDC event always receives rank 1.
    window = Window.partitionBy("loan_id").orderBy(col("_updated_at").desc())

    # 2. Add the rank column ONCE
    # This creates a single "Parent" plan for both branches
    df_ranked = df_updated.withColumn("rn", row_number().over(window))

    # --- Branch Logic ---
    # 3. Clean: The absolute latest truth (rn=1) for the Silver Layer MERGE
    df_clean = df_ranked.filter(col("rn") == 1).drop("rn")

    # 4. Bad/Stale: Older intermediate updates (rn>1) for the Audit/Rejected table
    df_bad = df_ranked.filter(col("rn") > 1).drop("rn")

    # --- EXECUTE YOUR WRITES HERE ---    
    # 5. Register Clean Source View for SQL transformations
    df_clean.createOrReplaceTempView("repayments_batch_updates")

    merge_query = f"""
    MERGE INTO {REPAYMENTS_SILVER} AS target
        USING repayments_batch_updates AS source
        ON target.loan_id = source.loan_id
        WHEN MATCHED AND source._bronze_version > target._bronze_version THEN
          UPDATE SET
            target.total_principal_received = source.total_principal_received,
            target.total_interest_received = source.total_interest_received,
            target.total_late_fee_received = source.total_late_fee_received,
            target.total_payment_received = source.total_payment_received,
            target.last_payment_amount = source.last_payment_amount,
            target.last_payment_date = source.last_payment_date,
            target.next_payment_date = source.next_payment_date,
            target._ingested_at = source._ingested_at,
            target._source_file = source._source_file,
            target._bronze_version = source._bronze_version,
            target._updated_at = source._updated_at,
            target._batch_id = source._batch_id          
        WHEN NOT MATCHED THEN
          INSERT (
            loan_id, total_principal_received, total_interest_received, total_late_fee_received,
            total_payment_received, last_payment_amount, last_payment_date, next_payment_date,
            _ingested_at, _source_file, _bronze_version, _updated_at, _batch_id
          )
          VALUES (
            source.loan_id, source.total_principal_received, source.total_interest_received, source.total_late_fee_received,
            source.total_payment_received, source.last_payment_amount, source.last_payment_date, source.next_payment_date,
            source._ingested_at, source._source_file, source._bronze_version, source._updated_at, _batch_id
          )
    """
    spark_session.sql(merge_query)


# Execute trigger-once pipeline for batch-like efficiency on streaming logic.
query_repayments = (
    df_repayments_final.writeStream
    .outputMode("append") 
    .option("checkpointLocation", SILVER_CHECKPOINT_REPAYMENTS)
    .trigger(availableNow=True)
    .foreachBatch(upsert_repayments_to_silver)
    .start()
)

# block notebook termination until micro-batch is committed
query_repayments.awaitTermination()


In [0]:
%sql
select count(1) from lending_risk.silver.repayments;
-- select * from lending_risk.silver.repayments 
--limit 3;
-- 3332
-- 6599
-- 9881